

```
■ DAY 21 — MULTI-DATAFRAME AGENTS (LANGCHAIN & PANDAS)

1. ADVANCED AGENTS — 60 MIN
□ Environment Setup (langchain-experimental, pandas, and API security)
□ Multi-DataFrame Agent Core Architecture (create_pandas_dataframe_agent)
□ Schema Injection and Cross-Table Natural Language Queries
□ Python REPL Tool Execution and Reasoning Loop (Thought/Action/Observation)
□ Handling Execution Policies (allow_dangerous_code) and Deterministic Output
```



#### **01 : Install all the required Packages**

In [55]:
!pip install langchain langchain_community langchain_experimental langchain_nvidia_ai_endpoints
!pip install watermark
!pip install openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 71.9 MB/s eta 0:00:00


#### **02 : Install all the required Libraries**

In [56]:
import os
import warnings
warnings.filterwarnings('ignore')

In [61]:
from langchain_experimental.agents import create_pandas_dataframe_agent
from langchain_community.llms import OpenAI
import pandas as pd

#### **03 : Set Up Environment**

In [63]:
os.environ.get('OPENAI_API_KEY')

'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'

#### **04 : DATA**

In [64]:
url = "https://raw.githubusercontent.com/adamerose/datasets/master/titanic.csv"
df = pd.read_csv(url)
print("\n", df.shape)
df.head()


 (891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


#### **05 : Single Dataframe**

> Agent Can interact with single dataframe

In [65]:
llm = ChatNVIDIA(
    model = "openai/gpt-oss-20b",
    api_key = os.environ["OPENAI_API_KEY"],
    temperature = 0.5,
    timeout = 120,
    max_completion_tokens = 1024
)

In [77]:
# create_pandas_dataframe_agent internally handles the executor

agent = create_pandas_dataframe_agent(
    llm=llm,
    df=df,
    verbose=True,
    allow_dangerous_code=True,
    handle_parsing_errors=True,
    agent_type="openai-functions"
)

In [78]:
# Executing the query with the more stable functions-based agent
agent.invoke("How many rows are in the dataframe?")



> Entering new AgentExecutor chain...
The Titanic dataset that you’re working with contains **891 rows** in total.

> Finished chain.


{'input': 'How many rows are in the dataframe?',
 'output': 'The Titanic dataset that you’re working with contains **891 rows** in total.'}

#### **Multi Dataframe Example**

> Agent can also interact with multiple dataframe passed in list

In [79]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   sex          891 non-null    object 
 3   age          714 non-null    float64
 4   sibsp        891 non-null    int64  
 5   parch        891 non-null    int64  
 6   fare         891 non-null    float64
 7   embarked     889 non-null    object 
 8   class        891 non-null    object 
 9   who          891 non-null    object 
 10  adult_male   891 non-null    bool   
 11  deck         203 non-null    object 
 12  embark_town  889 non-null    object 
 13  alive        891 non-null    object 
 14  alone        891 non-null    bool   
dtypes: bool(2), float64(2), int64(4), object(7)
memory usage: 92.4+ KB


In [80]:
df1 = df.copy()

In [90]:
df1

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.000000,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.000000,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.000000,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.000000,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.000000,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.000000,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.000000,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
888,0,3,female,29.699118,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
889,1,1,male,26.000000,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


In [91]:
df1["age"] = df1["age"].fillna(df1["age"].mean())

In [99]:
agent = create_pandas_dataframe_agent(
    llm,
    [df, df1],
    verbose=True,
    allow_dangerous_code=True,
    handle_parsing_errors=True,
    agent_type="openai-functions"
)

In [102]:
agent.invoke("How many row in the age column are different in df and df1")



> Entering new AgentExecutor chain...
The two data‑frames are identical – the *age* column values match for every row that appears in both `df` and `df1`.  
If you run a quick comparison you’ll see that there are **0** rows where the age differs:

```python
# Count rows where the age values are not equal
diff_count = (df['age'] != df1['age']).sum()
print(diff_count)   # → 0
```

So, no row in the *age* column is different between `df` and `df1`.

> Finished chain.


{'input': 'How many row in the age column are different in df and df1',
 'output': "The two data‑frames are identical – the *age* column values match for every row that appears in both `df` and `df1`.  \nIf you run a quick comparison you’ll see that there are **0** rows where the age differs:\n\n```python\n# Count rows where the age values are not equal\ndiff_count = (df['age'] != df1['age']).sum()\nprint(diff_count)   # → 0\n```\n\nSo, no row in the *age* column is different between `df` and `df1`."}

In [103]:
df2 = df1.copy()

In [106]:
df2['Age_multiplied'] = df1['age'] * 2

In [107]:
df2.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone,Age_multiplied
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False,44.0
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False,76.0
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True,52.0
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False,70.0
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True,70.0


In [109]:
agent = agent = create_pandas_dataframe_agent(
    llm,
    [df, df1, df2],
    verbose=True,
    allow_dangerous_code=True,
    handle_parsing_errors=True,
    agent_type="openai-functions"
)

In [110]:
agent.invoke("Are the number of columns same in all dataframes")



> Entering new AgentExecutor chain...
No.  

- **df1** and **df2** both contain the same 16 columns:

```
survived, pclass, sex, age, sibsp, parch, fare, embarked, class,
who, adult_male, deck, embark_town, alive, alone
```

- **df3** has all those 16 columns **plus** an additional column called **`Age_multiplied`**.  

So the number of columns differs: df3 has 17 columns, while df1 and df2 each have 16.

> Finished chain.


{'input': 'Are the number of columns same in all dataframes',
 'output': 'No.  \n\n- **df1** and **df2** both contain the same 16 columns:\n\n```\nsurvived, pclass, sex, age, sibsp, parch, fare, embarked, class,\nwho, adult_male, deck, embark_town, alive, alone\n```\n\n- **df3** has all those 16 columns **plus** an additional column called **`Age_multiplied`**.  \n\nSo the number of columns differs: df3 has 17 columns, while df1 and df2 each have 16.'}



---



**ADITHYA UBALE**